# Classificação de Imagens

Referência
- [TensorFlow: cifar10](https://www.tensorflow.org/datasets/catalog/cifar10)

## Imports

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import keras
import torch

## Carregar Dataset

In [ ]:
(x_training, y_training), (x_test, y_test) = keras.datasets.cifar10.load_data()

## Sobre os Dados

### Pré-processamento

In [ ]:
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]


print("Train images:", len(x_training))
print("Test images:", len(x_test))


# Converter para `float32`:
x_training = x_training.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# "Achatar" os dados:
y_training = y_training.flatten()
y_test = y_test.flatten()

### Visualização

In [ ]:
plt.figure(figsize=(8,8))

for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_training[i])
    plt.title(classes[y_training[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

## Sobre o Modelo

### Montar

In [ ]:
model = keras.Sequential([
  keras.layers.Input(shape=(32,32,3)),

  keras.layers.Conv2D(32, (3,3), activation="relu", padding="same"),
  keras.layers.MaxPooling2D(2,2),

  keras.layers.Conv2D(64, (3,3), activation="relu", padding="same"),
  keras.layers.MaxPooling2D(2,2),

  keras.layers.Flatten(),
  keras.layers.Dense(128, activation="relu"),
  keras.layers.Dropout(0.5),

  keras.layers.Dense(10, activation="softmax"),
])


model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


model.summary()

### Treinamento

#### Alocar GPU

O treinamento roda na GPU do Apple Silicon (Mac mini M4) por meio do
backend **PyTorch** do Keras 3, que utiliza o dispositivo **MPS**
(Metal Performance Shaders).

A alocação do dispositivo é automática — o backend PyTorch seleciona o
MPS quando disponível — e a célula seguinte apenas confirma qual
dispositivo está em uso. Para forçar a CPU em uma comparação, defina
`KERAS_TORCH_DEVICE="cpu"` junto com `KERAS_BACKEND`.

In [ ]:
os.environ["KERAS_BACKEND"] = "torch"
device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Keras backend: {keras.backend.backend()}")
print(f"Training device: {device}")

#### Treinar

In [ ]:
logs = model.fit(
    x_training,
    y_training,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
)

### Avaliar

In [ ]:
_, acc = model.evaluate(x_test, y_test)

print("Acc:", acc)

### Fazer Predição

In [ ]:
img = x_test[0:1]
label = classes[y_test[0]]

In [ ]:
predict = model.predict(img)
expected_class = classes[np.argmax(predict[0])]
confidence = predict[0].max()


print(f"Real class: {label} | Expected class: {expected_class} | (Confidence: {confidence:.2%})")


plt.figure()
plt.imshow(img[0])
plt.title(f"Real: {label} | Expected: {expected_class}")
plt.axis("off")
plt.show()